## Natural language inference

### Contradictory dear watson
***

Project: https://www.kaggle.com/competitions/contradictory-my-dear-watson

### Starter guide

https://keras.io/guides/keras_nlp/getting_started/

#### Importar liberias

In [2]:
import numpy as np 
import pandas as pd 
import tensorflow as tf
from tensorflow import keras
import keras_nlp
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

E0000 00:00:1730377354.098251    3465 common_lib.cc:798] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:479
D1031 12:22:34.106811769    3465 config.cc:196]                        gRPC EXPERIMENT call_status_override_on_cancellation   OFF (default:OFF)
D1031 12:22:34.106829808    3465 config.cc:196]                        gRPC EXPERIMENT call_v3                                OFF (default:OFF)
D1031 12:22:34.106833237    3465 config.cc:196]                        gRPC EXPERIMENT canary_client_privacy                  ON  (default:ON)
D1031 12:22:34.106835583    3465 config.cc:196]                        gRPC EXPERIMENT capture_base_context                   ON  (default:ON)
D1031 12:22:34.106837929    3465 config.cc:196]                        gRPC EXPERIMENT client_idleness                        ON  (defau

TensorFlow version: 2.16.1
KerasNLP version: 0.15.1


In [3]:
import warnings
warnings.filterwarnings('ignore')

#### Conf

In [4]:
try:
    # detect and init the TPU
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    print("All devices: ", tf.config.list_logical_devices('TPU'))
except ValueError:
    strategy = tf.distribute.get_strategy()  # default strategy if no TPU available

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local


I0000 00:00:1730377360.533560    3465 service.cc:145] XLA service 0x56f67573b3b0 initialized for platform TPU (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1730377360.533613    3465 service.cc:153]   StreamExecutor device (0): TPU, 2a886c8
I0000 00:00:1730377360.533617    3465 service.cc:153]   StreamExecutor device (1): TPU, 2a886c8
I0000 00:00:1730377360.533620    3465 service.cc:153]   StreamExecutor device (2): TPU, 2a886c8
I0000 00:00:1730377360.533623    3465 service.cc:153]   StreamExecutor device (3): TPU, 2a886c8
I0000 00:00:1730377360.533626    3465 service.cc:153]   StreamExecutor device (4): TPU, 2a886c8
I0000 00:00:1730377360.533628    3465 service.cc:153]   StreamExecutor device (5): TPU, 2a886c8
I0000 00:00:1730377360.533631    3465 service.cc:153]   StreamExecutor device (6): TPU, 2a886c8
I0000 00:00:1730377360.533634    3465 service.cc:153]   StreamExecutor device (7): TPU, 2a886c8


INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:5, TPU, 0, 0)
I

In [5]:
RESULT_DICT = {
    0 : "entailment",
    1 : "neutral",
    2 : "contradiction"
}

#### Data

In [6]:
df_train_data = pd.read_csv('/kaggle/input/contradictory-my-dear-watson/train.csv')
df_submit_data = pd.read_csv('/kaggle/input/contradictory-my-dear-watson/test.csv')

In [7]:
df_train, df_test = train_test_split(df_train_data, test_size=0.2, random_state=42)

### Intermediate model

**Fine tuning a pretrained BERT backbone**

#### Data prep

In [8]:
# Asegúrate de que 'premise' y 'hypothesis' sean listas de strings
premises = df_train['premise'].astype(str).tolist()
hypothesis = df_train['hypothesis'].astype(str).tolist()

# Define las labels (etiquetas) para entrenar
labels = tf.convert_to_tensor(df_train['label'])

# Verifica que todas las entradas tengan la misma longitud
assert len(premises) == len(hypothesis) == len(labels), "Las dimensiones de las entradas no coinciden."

# Crea un dataset a partir de las entradas
train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypothesis), labels)).batch(32)


#### Define model

In [9]:
# Define el modelo dentro del scope de la estrategia
with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi",
        num_classes=3
    )
    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

I0000 00:00:1730377366.892436    3465 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


#### Train model

In [10]:
# classifier.fit(train_dataset, epochs=10)

### Advanced model

**Fine tuning with user-controlled preprocessing**

#### Data prep

Separate preprocessing from the same preset

Each model architecture has a parallel preprocessor Layer with its own from_preset constructor. Using the same preset for this Layer will return the matching preprocessor as the task.

In [11]:
# Asegúrate de que 'premise' y 'hypothesis' sean listas de strings
premises = df_train['premise'].astype(str).tolist()
hypotheses = df_train['hypothesis'].astype(str).tolist()
labels = tf.convert_to_tensor(df_train['label'])

assert len(premises) == len(hypotheses) == len(labels), "Las dimensiones de las entradas no coinciden."

train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))

In [14]:
sample_premise = premises[0]  # Usa un ejemplo de tu lista
sample_hypothesis = hypotheses[0]

**Preprocesador Bert**

In [15]:
# Definir el preprocesador
preprocessor = keras_nlp.models.BertPreprocessor.from_preset(
    "bert_base_multi",
    sequence_length=512,
)

In [16]:
# Preprocesar un solo ejemplo
output = preprocessor(sample_premise, sample_hypothesis)

# Extraer los valores
input_ids = output[0]['token_ids'].numpy()  # Convierte el tensor a un array de NumPy
segment_ids = output[0]['segment_ids'].numpy()  # Convierte el tensor a un array de NumPy
attention_mask = output[0]['padding_mask'].numpy()  # Convierte el tensor a un array de NumPy

# Imprimir los resultados
print("Input IDs:", input_ids)
print("Segment IDs:", segment_ids)
print("Attention Mask:", attention_mask)


Input IDs: [  101 76295   763 28089 10429   770 25473 11242 19249 16152   119   102
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0    

In [17]:
train_cached = (
    train_dataset.map(lambda x, y: (preprocessor(x[0], x[1]), y), num_parallel_calls=tf.data.AUTOTUNE)
                  .cache()
                  .prefetch(tf.data.AUTOTUNE)
)


#### Define model


- tf.keras.optimizers.Adam(): Este es un optimizador que se adapta durante el entrenamiento y es popular por su rendimiento en muchos problemas de aprendizaje profundo.


- tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True): Esta función se utiliza para medir la discrepancia entre las predicciones del modelo y las etiquetas verdaderas. El parámetro from_logits=True indica que las salidas del modelo son logits (valores sin pasar por una función de activación como softmax), lo cual es común en problemas de clasificación.


- metrics=['accuracy']: Esto permite que el modelo calcule y muestre la precisión durante el entrenamiento y la validación. Es una medida común de rendimiento en problemas de clasificación.

In [18]:
df_train.head()

,id,premise,hypothesis,lang_abv,language,label
5289,4e5ad9e03a,اب اتنا خفیہ تھا یہ.,یہ عوامی معلومات تھی۔,ur,Urdu,2
6647,fea0d3c7e8,oh that's accommodating,That is disruptive.,en,English,2
1245,4ce586b61e,more than anything else in this day and age th...,"In your decisions age is a big factor, and I a...",en,English,1
4417,0dbcd1012b,"Aie! les boucaniers en-dessous s'écriaient, et...",Les Buccaneers étaient bruyants quand ils disa...,fr,French,0
5662,157895e59c,"一个采购战略组织，是壮大的力资本发展战略的一部分, 这是在原则VI上讨论的。",原则四涉及财富500强机构的资本发展战略。,zh,Chinese,1


In [19]:
# Preparar los datos
premises = df_train['premise'].astype(str).tolist()
hypotheses = df_train['hypothesis'].astype(str).tolist()
labels = tf.convert_to_tensor(df_train['label'].tolist())

In [22]:
# me parece que esta funcion es la que rompe todo

# Procesar los textos y obtener los IDs, las máscaras de atención y los IDs de segmento
def preprocess_data(premise, hypothesis):
    # Usar el preprocesador
    outputs = preprocessor(premise, hypothesis)
    
    # Convertir a NumPy para manejar como arrays
    input_ids = outputs[0].numpy()  # Primer elemento de la tupla
    attention_mask = outputs[1].numpy()  # Segundo elemento de la tupla
    segment_ids = outputs[2].numpy()  # Tercer elemento de la tupla
    
    return (input_ids, attention_mask, segment_ids)


In [23]:

# Crear el dataset
train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))

# Preprocesar los datos y generar el dataset final
train_cached = (
    train_dataset.map(lambda x, y: (preprocessor(x[0], x[1]), y), num_parallel_calls=tf.data.AUTOTUNE)
                  .cache()
                  .prefetch(tf.data.AUTOTUNE)
)


In [24]:
# Usar strategy.scope() si estás utilizando estrategia de distribución
with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi",
        preprocessor=None,  # Ya manejamos el preprocesador antes
        num_classes=3  # Asegúrate de que esto coincide con tus datos
    )

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # Entrenar el modelo
    classifier.fit(
        train_cached,
        epochs=3,
    )


IndexError: list index out of range

In [ ]:
premises = df_train['premise'].astype(str).tolist()
hypotheses = df_train['hypothesis'].astype(str).tolist()
labels = tf.convert_to_tensor(df_train['label'])

train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))

# Preprocesar los datos
preprocessor = keras_nlp.models.BertPreprocessor.from_preset("bert_base_multi", sequence_length=512)

In [ ]:
# Aplicar el preprocesador a cada ejemplo en el conjunto de datos
train_cached = (
    train_dataset.map(lambda x, y: preprocessor(x[0], x[1]), num_parallel_calls=tf.data.AUTOTUNE)
                  .cache()
                  .prefetch(tf.data.AUTOTUNE)
)


In [ ]:

# Usar strategy.scope() si estás utilizando estrategia de distribución
with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi",
        preprocessor=None,
        num_classes=3  # Asegúrate de que esto coincide con tus datos
    )

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # Entrenar el modelo
    classifier.fit(
        train_cached,
        epochs=3,
    )

In [ ]:
for example in train_dataset.take(1):
    print(example)


In [ ]:
import tensorflow as tf
import keras_nlp

# Prepara tus datos (asegúrate de que esto esté hecho antes de la compilación)
premises = df_train['premise'].astype(str).tolist()
hypotheses = df_train['hypothesis'].astype(str).tolist()
labels = tf.convert_to_tensor(df_train['label'])

train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))

# Preprocesar los datos
preprocessor = keras_nlp.models.BertPreprocessor.from_preset("bert_base_multi", sequence_length=512)
train_cached = (
    train_dataset.map(preprocessor, num_parallel_calls=tf.data.AUTOTUNE)
                  .cache()
                  .prefetch(tf.data.AUTOTUNE)
)

# Usar strategy.scope() si estás utilizando estrategia de distribución
with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi",
        preprocessor=None,
        num_classes=3  # Asegúrate de que esto coincide con tus datos
    )

    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    # Entrenar el modelo
    classifier.fit(
        train_cached,
        epochs=3,
    )


In [ ]:
with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi", 
        preprocessor = None, 
        num_classes=3  # Cambia esto según tus datos
    )
    
    classifier.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

    classifier.fit(
    train_cached,
    epochs=3,
)

#### Train model

In [ ]:
classifier.fit(
    train_cached,
    epochs=3,
)

In [ ]:
# Asegúrate de que 'premise' y 'hypothesis' sean listas de strings
premises = df_train['premise'].astype(str).tolist()
hypotheses = df_train['hypothesis'].astype(str).tolist()
labels = tf.convert_to_tensor(df_train['label'])

assert len(premises) == len(hypotheses) == len(labels), "Las dimensiones de las entradas no coinciden."

train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))

preprocessor = keras_nlp.models.BertPreprocessor.from_preset(
    "bert_base_multi",
    sequence_length=512,
)

train_cached = (
    train_dataset
    .map(lambda x, y: (
        preprocessor(x[0]), 
        preprocessor(x[1]),
        y
    ), num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi", 
        num_classes=3
    )
    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )




In [ ]:
classifier.fit(
    train_cached,
    epochs=3,
)

In [ ]:
# Asegúrate de que 'premise' y 'hypothesis' sean listas de strings
premises = df_train['premise'].astype(str).tolist()
hypotheses = df_train['hypothesis'].astype(str).tolist()
labels = tf.convert_to_tensor(df_train['label'])

assert len(premises) == len(hypotheses) == len(labels), "Las dimensiones de las entradas no coinciden."

train_dataset = tf.data.Dataset.from_tensor_slices(((premises, hypotheses), labels))

preprocessor = keras_nlp.models.BertPreprocessor.from_preset(
    "bert_base_multi",
    sequence_length=512,
)

train_cached = (
    train_dataset
    .map(lambda x, y: (
        preprocessor(x[0]),  # premisas
        preprocessor(x[1]),  # hipótesis
        y  # etiquetas
    ), num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

with strategy.scope():
    classifier = keras_nlp.models.BertClassifier.from_preset(
        "bert_base_multi", 
        num_classes=3
    )
    classifier.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )



In [ ]:
print(type(x[0]), x[0])  # Debe ser una cadena
print(type(x[1]), x[1])  # Debe ser una cadena


In [ ]:
classifier.fit(
    train_cached,
    epochs=3,
)


In [ ]:
#### Data prep b

Custom preprocessing

In cases where custom preprocessing is required, we offer direct access to the Tokenizer class that maps raw strings to tokens. It also has a from_preset() constructor to get the vocabulary matching pretraining